# Fake News Detector

A simple, end-to-end ML pipeline that classifies news article text as **FAKE** or **REAL**.

**Pipeline:** raw text → TF-IDF vectorization → classifier (Logistic Regression / Passive Aggressive) → prediction

This notebook uses the **Kaggle Fake and Real News dataset** (`Fake.csv` / `True.csv`, ~45k articles) with known dataset leakage issues handled (see Section 2).

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pd.set_option('display.max_colwidth', 80)

## 2. Load the dataset

Using the Kaggle **Fake and Real News Dataset** (`Fake.csv` / `True.csv`, ~45k articles total).

⚠️ **Important — this dataset has several known leakage issues, all handled below:**
- `date` perfectly separates the two classes and is dropped entirely as a feature.
- ~99% of REAL articles start with a wire-service dateline like `WASHINGTON (Reuters) -`, and the word "Reuters" also leaks into article bodies (e.g. "told Reuters"). We strip both.
- Roughly a third of FAKE articles end with a `Photo by X/Getty Images` credit line or contain `pic.twitter.com` embed links that never appear in REAL articles — these are scraping artifacts, not content. We strip them too.
- We **keep** the `subject` column (e.g. `politicsNews`, `left-news`) but never feed it to the model — instead we use it later, in Section 10, to build a genuinely harder train/test split across different sources.

In [ ]:
import re as _re

fake = pd.read_csv('Fake.csv')
real = pd.read_csv('True.csv')

fake['label'] = 1  # FAKE
real['label'] = 0  # REAL

df = pd.concat([fake, real], ignore_index=True)
df = df.drop(columns=['date'], errors='ignore')  # keep 'subject' for later; drop only 'date' here

# Strip wire-service dateline boilerplate (e.g. "WASHINGTON (Reuters) - ")
dateline_pattern = _re.compile(r'^.{0,80}?\(Reuters\)\s*-?\s*')
df['text'] = df['text'].astype(str).apply(lambda t: dateline_pattern.sub('', t, count=1))

# Strip remaining leaky artifacts: stray "Reuters" mentions, photo credits, embed links
df['text'] = df['text'].apply(lambda t: _re.sub(r'reuters', '', t, flags=_re.IGNORECASE))
df['text'] = df['text'].apply(lambda t: _re.sub(r'photo by.*?(getty images|images)\.?', '', t, flags=_re.IGNORECASE))
df['text'] = df['text'].apply(lambda t: _re.sub(r'featured image (is |via ).*', '', t, flags=_re.IGNORECASE))
df['text'] = df['text'].apply(lambda t: _re.sub(r'pic\.twitter\.com/\S+', '', t))
df['text'] = df['text'].apply(lambda t: _re.sub(r'https?://\S+|www\.\S+', '', t))

# Combine title + text as the full document
df['text'] = df['title'].astype(str) + ' ' + df['text'].astype(str)
df = df.drop(columns=['title'])

# Drop empty/near-empty rows, dedupe, and shuffle
df = df[df['text'].str.strip().str.len() > 20]
df = df.drop_duplicates(subset='text')
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(df['label'].value_counts())
print(f"Total articles: {len(df)}")
df.head()

## 3. Text cleaning

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)          # remove URLs
    text = re.sub(r'\d+', '', text)                       # remove digits
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()               # collapse whitespace
    return text

df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']].head()

## 4. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'],
    test_size=0.25, random_state=42, stratify=df['label']
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 5. TF-IDF vectorization

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.9, min_df=5, max_features=20000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

## 6. Train a classifier

We'll try Logistic Regression and Passive Aggressive Classifier (a common choice in fake-news literature) and compare.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Passive Aggressive (SGD)": SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0, max_iter=1000),
    "Multinomial Naive Bayes": MultinomialNB()
}

results = {}
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f"{name}: accuracy = {acc:.2f}")

## 7. Evaluate the best model in detail

In [ ]:
best_name = max(results, key=results.get)
best_model = models[best_name]
print(f"Best model: {best_name}\n")

preds = best_model.predict(X_test_tfidf)
print(classification_report(y_test, preds, target_names=['REAL', 'FAKE'], zero_division=0))

cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['REAL', 'FAKE'])
disp.plot(cmap='Blues')
plt.title(f"Confusion Matrix - {best_name}")
plt.show()

## 8. Predict on new text

In [ ]:
def predict_news(text, model=best_model, vectorizer=vectorizer):
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    label = "FAKE" if pred == 1 else "REAL"

    # Confidence score, if the model supports it
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(vec)[0]
        confidence = max(proba)
        return f"{label} (confidence: {confidence:.2f})"
    else:
        return label

# Try it out
test_headlines = [
    "Local hospital opens new pediatric wing after fundraising campaign.",
    "Scientists discover secret alien base hidden under the ocean, government denies.",
]

for headline in test_headlines:
    print(f"'{headline}'\n -> {predict_news(headline)}\n")

## 9. A harder, more honest test: cross-subject generalization

The random train/test split above still mixes the *same* subject categories into both train and test — so the model can partly succeed just by recognizing subject-specific phrasing it already saw examples of. A more honest test: train on **some** subjects and evaluate on **entirely different, unseen** subjects. This simulates what actually matters — can the model catch fake news from a source style it has never seen?

- **Train subjects:** `politicsNews` (REAL) + `News`, `politics` (FAKE)
- **Held-out test subjects:** `worldnews` (REAL) + `left-news`, `Government News`, `US_News`, `Middle-east` (FAKE) — none of these appear in training at all

In [ ]:
train_subjects_real = ['politicsNews']
train_subjects_fake = ['News', 'politics']
test_subjects_real = ['worldnews']
test_subjects_fake = ['left-news', 'Government News', 'US_News', 'Middle-east']

train_mask = (
    ((df['label'] == 0) & (df['subject'].isin(train_subjects_real))) |
    ((df['label'] == 1) & (df['subject'].isin(train_subjects_fake)))
)
test_mask = (
    ((df['label'] == 0) & (df['subject'].isin(test_subjects_real))) |
    ((df['label'] == 1) & (df['subject'].isin(test_subjects_fake)))
)

cs_train = df[train_mask]
cs_test = df[test_mask]
print(f"Cross-subject train: {len(cs_train)} articles {cs_train['label'].value_counts().to_dict()}")
print(f"Cross-subject test (unseen subjects): {len(cs_test)} articles {cs_test['label'].value_counts().to_dict()}")

cs_vectorizer = TfidfVectorizer(stop_words='english', max_df=0.9, min_df=5, max_features=20000, ngram_range=(1,2))
cs_X_train = cs_vectorizer.fit_transform(cs_train['clean_text'])
cs_X_test = cs_vectorizer.transform(cs_test['clean_text'])

cs_model = LogisticRegression(max_iter=1000)
cs_model.fit(cs_X_train, cs_train['label'])
cs_preds = cs_model.predict(cs_X_test)

print(f"\nCross-subject held-out accuracy: {accuracy_score(cs_test['label'], cs_preds):.3f}")
print(classification_report(cs_test['label'], cs_preds, target_names=['REAL', 'FAKE'], zero_division=0))

Notice the gap: accuracy on unseen subjects is meaningfully lower than the random-split number from Section 7, and FAKE-class precision in particular tends to drop — the model false-positives more on real articles it hasn't seen a similar style of before. **This gap is the honest measure of how well the model generalizes**, and it's the number to trust more than the one in Section 7.

## 10. Remaining limitations — read before trusting either number

Even the cross-subject accuracy above is not a clean measure of "truthfulness detection":

1. **Still single-corpus.** Every REAL article here is Reuters-style wire copy; every FAKE article comes from one of a handful of specific hyperpartisan sites. The model may still be learning *"formal AP/Reuters journalistic register"* vs *"opinionated blog-style writing"* rather than truth vs. falsehood — a real fact stated in a casual tone could still be flagged FAKE, and a false claim written in a neutral wire-style tone could pass as REAL.
2. **It doesn't fact-check.** There's no access to external facts, sources, or evidence — it's entirely pattern-matching on writing style.
3. **Time and topic drift.** This dataset is from 2016–2017 US politics; expect performance to degrade on other time periods, topics, or countries.
4. **Best real-world sanity check:** run `predict_news()` on recent headlines from outlets *not* in this dataset at all (e.g. paste in a BBC, AP, or non-English-language headline) — that tells you more than either accuracy number above.

For something closer to genuine fact-checking, you'd want to combine this with source-credibility databases, live fact-checking APIs, or an LLM-based approach that reasons about claims rather than classifying writing style.